In [ ]:
import json
import os
import subprocess
from pathlib import Path
from typing import Any, overload, Mapping, List, Dict, Iterable, Optional
from dataclasses import dataclass
from ollama import ResponseError 
from langchain_core.messages import (
    AIMessage,
    HumanMessage,
    SystemMessage,
    ToolMessage,
    BaseMessage
)
from langchain_core.tools import tool
from langchain_ollama import ChatOllama




In [ ]:
OLLAMA_BASE_URL = os.getenv(
    "OLLAMA_BASE_URL",
    "http://10.42.0.192:11434/",
)

MODEL_NAME = os.getenv(
    "OLLAMA_MODEL",
    "gpt-oss:20b",
)

llm = ChatOllama(
    model=MODEL_NAME,
    base_url=OLLAMA_BASE_URL,
    temperature=0.01,
    num_ctx=40960,
)

print(f"Using {MODEL_NAME} at {OLLAMA_BASE_URL}")

In [ ]:
@dataclass
class ToolEvent:
    iteration: int
    tool: str
    args: dict
    result: str

In [ ]:
events: list[ToolEvent] = []

In [ ]:
response = llm.invoke("Reply with exactly: Ollama connection works")
print(response.content)


In [ ]:
WORKSPACE = Path("./graphdb-project/iteration_3")
WORKSPACE.mkdir(parents=True, exist_ok=True)

WORKSPACE = WORKSPACE.resolve()
print(WORKSPACE)

In [ ]:
def safe_path(relative_path: str) -> Path:
    """
    Resolve a user-provided path inside WORKSPACE.
    Prevents paths such as ../../etc/passwd.
    """
    path = (WORKSPACE / relative_path).resolve()

    if path != WORKSPACE and WORKSPACE not in path.parents:
        raise ValueError(f"Path escapes workspace: {relative_path}")

    return path

In [ ]:
@tool
def search_file(path: str, query: str) -> str:
    """
    Search a local file for an **exact** text match.

    Parameters
    ----------
    path : str
        Path to the file to search (relative to the workspace or absolute).
    query : str
        The exact text to look for (case‑sensitive).

    Returns
    -------
    str
        JSON array of objects, each with:
            * `line` – 1‑based line number
            * `text` – the line content (trimmed of trailing newline)

        If the file is not found or is not a regular file,
        a short error message is returned instead of JSON.
        When no matches are found an empty JSON array (`[]`) is returned.

    Notes
    -----
    * The file is read with UTF‑8 encoding; if decoding fails,
      the fallback encoding `latin‑1` is used.
    * To keep the agent’s context small, the search stops after
      `max_hits` matches (default 10).
    * Very large files (over `max_chars` bytes) are truncated before
      searching; the truncated text ends with `"\n...[truncated]"`.
    """

    # Resolve path safely
    file_path = safe_path(path)

    # Basic file checks
    if not file_path.exists():
        return f"File does not exist: {path}"
    if not file_path.is_file():
        return f"Not a file: {path}"

    # Read the file (UTF‑8, fallback to latin‑1)
    try:
        content = file_path.read_text(encoding="utf-8")
    except UnicodeDecodeError:
        content = file_path.read_text(encoding="latin-1")

    # Avoid flooding the model with huge inputs
    max_chars = 30_000
    if len(content) > max_chars:
        content = content[:max_chars] + "\n...[truncated]"

    # Search for the exact query
    matches: List[Dict[str, str]] = []
    max_hits = 10
    for i, line in enumerate(content.splitlines(), start=1):
        if query in line:
            matches.append({"line": i, "text": line.strip()})
            if len(matches) >= max_hits:
                break

    # Return JSON (empty array if no matches)
    return json.dumps(matches, ensure_ascii=False, indent=2)


In [ ]:
@tool
def list_files() -> str:
    """List files and directories in the current project workspace."""
    entries = []

    for path in sorted(WORKSPACE.rglob("*")):
        relative = path.relative_to(WORKSPACE)
        if ".git" in relative.parts or "__pycache__" in relative.parts:
            continue

        suffix = "/" if path.is_dir() else ""
        entries.append(f"{relative}{suffix}")

    return "\n".join(entries) if entries else "(workspace is empty)"


In [ ]:
@tool
def read_file(
    path: str,
    line_start: Optional[int] = None,
    line_end: Optional[int] = None
) -> str:
    """
    Read a UTF‑8 text file from the project workspace.

    Parameters
    ----------
    path : str
        Path to the file (relative or absolute).
    line_start : int | None, default None
        First line to return (1‑based).  If omitted, start at the beginning.
    line_end : int | None, default None
        Last line to return (inclusive).  If omitted, go to the end.

    Returns
    -------
    str
        The requested content (possibly truncated to 30 k chars).  
        If the file does not exist or is not a file, a short error string is returned.
    """
    # Resolve path safely
    file_path = safe_path(path)

    # Basic checks
    if not file_path.exists():
        return f"File does not exist: {path}"
    if not file_path.is_file():
        return f"Not a file: {path}"

    # Read the file – try UTF‑8, fall back to latin‑1
    try:
        content = file_path.read_text(encoding="utf-8")
    except UnicodeDecodeError:
        content = file_path.read_text(encoding="latin-1")

    # Optionally slice by line numbers
    if line_start is not None or line_end is not None:
        lines = content.splitlines()
        # Normalize missing values
        start = line_start - 1 if line_start is not None else 0
        end   = line_end   if line_end   is not None else len(lines)
        # Guard against bad indices
        if start < 0 or end < start:
            return "Invalid line range specified."
        # Slice and re‑join
        content = "\n".join(lines[start:end])

    # Truncate very large output
    max_chars = 30_000
    if len(content) > max_chars:
        content = content[:max_chars] + "\n...[truncated]"

    return content


In [ ]:
from importlib.metadata import version

for package in [
    "langchain",
    "langchain-core",
    "langchain-ollama",
]:
    print(package, version(package))

In [ ]:
@tool
def write_file(path: str, content: Any, overwrite: bool = False) -> str:
    """
    Write `content` to `path`.  
    - If the file exists and `overwrite` is False (default), the call is a no‑op
      and you get a short “file exists” message.
    - If `overwrite` is True, the existing file is simply replaced.
    - `content` may be:
        • a mapping → pretty‑printed JSON (unless it contains a single string)
        • a string → written verbatim (real newlines, no `\\n`)

    Returns a human‑readable status message.
    """

    file_path = safe_path(path)

    if isinstance(content, Mapping):
        if "content" in content and isinstance(content["content"], str):
            content_str = content["content"]
        else:
            content_str = json.dumps(content, indent=2, ensure_ascii=False)
    elif isinstance(content, str):
        content_str = content
    else:
        raise TypeError(f"Unsupported content type {type(content)}")

    if file_path.exists() and not overwrite:
        return (
            f"File already exists: {path}. "
            "Use `overwrite=True` to replace it or use `read_file` + "
            "`edit_file` to modify it."
        )

    file_path.parent.mkdir(parents=True, exist_ok=True)
    file_path.write_text(content_str, encoding="utf-8")
    return (
        f"Created {len(content_str)} characters in "
        f"{file_path.relative_to(Path.cwd())}"
    )


In [ ]:
@tool
def edit_file(path: str, old_text: str, new_text: str) -> str:
    """
    Edit a UTF-8 text file by replacing one exact occurrence of old_text
    with new_text.

    The file must already exist. The replacement is intentionally limited
    to one occurrence so the agent cannot accidentally modify multiple
    unrelated sections.
    """
    file_path = safe_path(path)

    if not file_path.exists():
        return f"File does not exist: {path}"

    if not file_path.is_file():
        return f"Not a file: {path}"

    content = file_path.read_text(encoding="utf-8")

    occurrences = content.count(old_text)

    if occurrences == 0:
        return (
            f"Could not edit {path}: old_text was not found. "
            "Read the file again and use an exact text match."
        )

    if occurrences > 1:
        return (
            f"Could not edit {path}: old_text occurs {occurrences} times. "
            "Provide a larger, more specific old_text block."
        )

    updated_content = content.replace(old_text, new_text, 1)
    file_path.write_text(updated_content, encoding="utf-8")

    return (
        f"Edited {file_path.relative_to(WORKSPACE)}: "
        f"replaced {len(old_text)} characters with {len(new_text)} characters."
    )


In [ ]:
@tool
def run_command(command: str) -> str:
    """
    Run a non-interactive shell command inside the project workspace.
    Use this for formatting, tests, compilation, and inspection.
    """
    blocked_fragments = [
        "rm -rf",
        "shutdown",
        "reboot",
        "mkfs",
        "dd if=",
        ":(){",
        "curl | sh",
        "wget | sh",
    ]

    normalized = command.lower().replace(" ", "")
    for fragment in blocked_fragments:
        if fragment.replace(" ", "") in normalized:
            return f"Blocked potentially destructive command: {command}"

    try:
        result = subprocess.run(
            command,
            shell=True,
            cwd=WORKSPACE,
            capture_output=True,
            text=True,
            timeout=60,
            env={
                **os.environ,
                "PYTHONUNBUFFERED": "1",
            },
        )

        output = (
            f"exit_code: {result.returncode}\n"
            f"stdout:\n{result.stdout}\n"
            f"stderr:\n{result.stderr}"
        )

        if len(output) > 20_000:
            output = output[:20_000] + "\n...[output truncated]"

        return output

    except subprocess.TimeoutExpired:
        return "Command timed out after 60 seconds."
    except Exception as exc:
        return f"Command failed to run: {type(exc).__name__}: {exc}"


In [ ]:
TOOLS = [
    list_files,
    read_file,
    write_file,
    edit_file,
    search_file,
    run_command
]


TOOLS_BY_NAME = {tool.name: tool for tool in TOOLS}

for item in TOOLS:
    print(item.name)


In [ ]:
llm_with_tools = llm.bind_tools(TOOLS)


In [ ]:
response = llm_with_tools.invoke(
    "Use the list_files tool and report the files in the workspace."
)

print("content:", response.content)
print("tool calls:", response.tool_calls)

In [ ]:
DISTILLATION_PROMPT = """
The Year is 2026, You are Graph‑Partner, an AI collaborator specialized in distilling threads of work on a semantic‑web/OWL knowledge‑graph stack that runs locally on a Linux server with an NVIDIA RTX 5070 GPU.  

You are part of an agent pipeline that extracts facts from a thread

the next iteration will use your distillation to guide work and manage token context

you will be given the initial user prompt, followed by everything currently in the thread context, which may contain an earlier iteration of your distillation results as the third message in the series.

when the context already contains your earlier distillation result, the distillation may include information from messages that have been truncated from the context

if the third message decribes a tool call, we are on the first distillation pass for this thread

use the information from the context to infer the progress of the thread and help form the direction of the tool enabled agent

output **only** a single JSON object with these top‑level keys:
  - artifacts:   [{ "id":"", "type":"", "value":"" }]
  - claims:      [{ "statement":"", "source":"", "confidence":0‑1 }]
  - understandings:[{ "concept":"", "detail":""}]
  - hypotheses:  [{ "hypothesis":"", "status":"pending/confirmed/ruled‑out", "confidence":0‑1 }]
  - direction:   [{ "step":"", "deadline":"YYYY‑MM‑DD"}]

If a key has no entries, use an empty array.  
**Do NOT** wrap the output in Markdown or quotes around keys.  
Make sure the JSON is syntactically valid (no trailing commas, proper quoting).

"""

In [ ]:
SYSTEM_PROMPT = """
The Year is 2026, You are Graph‑Partner, an AI collaborator specialized in building, deploying, and iterating on a semantic‑web/OWL knowledge‑graph stack that runs locally on a Linux server with an NVIDIA RTX 5070 GPU.  

Your primary mission is to help the user (a 48‑year‑old software engineer) create a **self‑sustaining, compute‑backbone** that powers a “human + AI” ecosystem.  You must:

1. **Stay Technical, No Job‑Search Talk**  
   • Skip any corporate‑HR or job‑search advice.  
   • Focus on concrete tooling, code, and deployment steps.

2. **Lead with the OWL Inference Flow**  
   • Explain, build, and maintain an OWL ontology, RDF data, and a forward‑chaining reasoner.  
   • Show how to expose this through SPARQL and/or GraphQL, then hook it to a local LLM (Ollama / vLLM).  

3. **Give Hands‑On, Code‑Ready Guidance**  
   • Provide Docker‑Compose files, shell scripts, Python snippets, and Jena/GraphDB commands.  
   • Offer sanity‑check templates (e.g., “verify that inferred triples appear in SPARQL results”).  

4. **Maintain an Iterative Loop**  
   • After each step, ask a “quick check” question (e.g., “Did the reasoner add the inferred triple?”).  
   • Suggest metrics to capture (token‑rate, query latency, GPU utilisation) and how to log them.  

5. **Ask Clarifying Questions When Needed**  
   • If any environmental detail is missing (e.g., Docker version, data location, existing ontology files), ask for it.  
   • Do not bombard with questions; one or two targeted ones per turn are enough.

6. **Use a Friendly, Future‑Oriented Tone**  
   • Encourage experimentation, celebrate small wins, and keep the user motivated.
You are not allowed to declare success without reproducing and recording evidence..  

Current World Model:
- Inspect existing files.
- Use write_file to create example implementaions.
- use search_file to get section around search criteria
- Use edit_file for targeted modifications to existing files, modify anything you need to in the workspace.
- Use read_file to inspect relevant files before editing them.
- When using edit_file, provide an exact old_text match and a precise new_text replacement.
- If an edit_file operation fails because old_text was not found or is ambiguous, read the file again before retrying.
- Use run_command for tests, formatters, linters, compilers, and basic inspection.
- Do not claim that code works unless you actually run an appropriate check.
- Keep generated code focused and maintainable.
- exit for more information only when the requirement is genuinely ambiguous.
- Do not delete or overwrite unrelated files.
- All paths must be relative to the project workspace.
"""

In [ ]:
print(SYSTEM_PROMPT)

In [ ]:
# ------------------------------------------------------------------
# 0️⃣  Helpers
# ------------------------------------------------------------------
def truncate_history(
    messages: List[BaseMessage],
    max_tokens: int = 25_000,
    tokenizer=None,
    preserve: int = 2,
) -> List[BaseMessage]:
    """
    Return a *new* list that

    1. always keeps the first ``preserve`` messages (default 2),
    2. truncates the *remaining* messages so that the total token count
       (estimated or actual, depending on ``tokenizer``) does not exceed
       ``max_tokens``.

    The original message list is never mutated.

    Parameters
    ----------
    messages: List[BaseMessage]
        All messages in chronological order (index 0 is the oldest).
    max_tokens: int, default 25_000
        Token budget for the *truncated* part of the history.  
        Tokens used by the preserved messages are *not* counted.
    tokenizer: callable, optional
        If supplied, must accept a string and return an object whose
        ``__len__`` gives the token count.  This is useful when you
        want an exact count rather than the 1‑token ≈ 4‑chars heuristic.
    preserve: int, default 2
        Number of messages that must stay in the returned list regardless
        of their size.

    Returns
    -------
    List[BaseMessage]
        A new list containing the preserved messages followed by the
        truncated remainder.
    """
    # Defensive copy – we never mutate the input list
    msg_len = len(messages)

    # If everything fits, just return a shallow copy
    if msg_len <= preserve:
        return messages.copy()

    # 1️⃣ Keep the first `preserve` messages
    kept_first = messages[:preserve]

    # 2️⃣ Truncate the rest
    rest = messages[preserve:]
    total = 0
    kept_rest = []

    for msg in reversed(rest):
        # Count tokens for this message
        if tokenizer is not None:
            try:
                n = len(tokenizer(msg.content))
            except Exception:
                # Fallback to the simple heuristic if tokenizer fails
                n = len(msg.content) // 4
        else:
            n = len(msg.content) // 4

        if total + n > max_tokens:
            print("truncating")
            break

        kept_rest.append(msg)
        total += n

    # 3️⃣ Return the combined list (preserved first + truncated rest)
    return kept_first + list(reversed(kept_rest))


def run_agent(
    user_request: str,
    max_iterations: int = 150,
    verbose: bool = False,
) -> str:
    messages = [
        SystemMessage(content=SYSTEM_PROMPT),
        HumanMessage(content=user_request),
    ]

    for iteration in range(max_iterations):
        if verbose:
            print(f"\n--- iteration {iteration + 1} ---")

        try:
            response = llm_with_tools.invoke(messages)
        except ResponseError as e:
            # 1. Show the error to the LLM
    
            events.append(
                ToolEvent(
                    iteration=iteration + 1,
                    tool="error_response",
                    args={},
                    result=str(e)
                )
            )
            messages.append(
                SystemMessage(
                    content=f"⚠️  Parsing error: {e}. "
                            "Generate a correct tool call or explain the issue."
                )
            )
            # 2. Don't count this as a real iteration
            continue

        
        messages.append(response)

        events.append(
            ToolEvent(
                iteration=iteration + 1,
                tool="response",
                args={},
                result=str(response)
            )
        )

        print(len(messages))
        tool_calls = response.tool_calls or []

        if verbose:
            if response.content:
                print("Assistant:", response.content)
            print("Tool calls:", tool_calls)

        # The model is finished when it returns no tool calls.
        if not tool_calls:
            return {"condition" : "no tool calls",
                    "final_response": response.content,
                    "iterations": iteration + 1,
                    "events": events}


        if response.usage_metadata["input_tokens"] > 15000:   # tweak threshold
            # Build the raw conversation string
            conv_text = "\n".join(
                [msg.content for msg in messages if isinstance(msg, (HumanMessage, AIMessage, ToolMessage))
            ])
            # Replace the whole history with the extraction result
            distillation_messages = [
                SystemMessage(content=DISTILLATION_PROMPT),
                HumanMessage(content=conv_text),
            ]
            distilled = False
            for _ in range(3):
                try:
                    facts_json = llm.invoke(distillation_messages)
                    facts = json.loads(facts_json.content)
                    print("DISTILLATION SUCCESS")
                    print(facts)
                    distilled = True
                    break
                except Exception as exc:
                    print(exc)
                    print(facts_json)
            
            if distilled:
                # the extracted facts become the 3rd message, 
                if isinstance(messages[2], HumanMessage):
                    # either replace the old summary
                    messages = messages[:2]+[(HumanMessage(content=json.dumps(facts, indent=2)))]+messages[3:]
                else:
                    # or insert new summary at position 2
                    messages = messages[:2]+[(HumanMessage(content=json.dumps(facts, indent=2)))]+messages[2:]
    
                # prune older history to keep token budget low
                messages = truncate_history(messages, max_tokens=10_000, preserve=3)

        
        for tool_call in tool_calls:
            tool_name = tool_call["name"]
            tool_args = tool_call.get("args", {})
            tool_call_id = tool_call["id"]

            selected_tool = TOOLS_BY_NAME.get(tool_name)

            if selected_tool is None:
                tool_result = f"Unknown tool: {tool_name}"
            else:
                try:
                    tool_result = selected_tool.invoke(tool_args)
                except Exception as exc:
                    tool_result = (
                        f"Tool error: {type(exc).__name__}: {exc}"
                    )
                    
            events.append(
                ToolEvent(
                    iteration=iteration + 1,
                    tool=tool_name,
                    args=tool_args,
                    result=str(tool_result)
                )
            )
            
            if verbose:
                print(f"Executing: {tool_name}({tool_args})")
                print(str(tool_result)[:2_000])
            else:
                print(f"Executing: {tool_name}()")

            messages.append(
                ToolMessage(
                    content=str(tool_result),
                    tool_call_id=tool_call_id,
                )
            )



    return {"condition" : f"Agent stopped after {max_iterations} iterations. The workspace may contain partial results.",
            "final_response": response.content,
            "iterations": iteration + 1,
            "events": events} 


In [ ]:
for e in events:
    if e.tool == "response":
        print(e.result)

In [ ]:
request = '''
## 🎯 What to do next

Below is a *minimal‑yet‑complete* action plan that will:

1. **Fix the syntax bug** (`rdfs:` prefix) DONE BY YOU ON A PREVIOUS ATTEMPT
2. **Make the Turtle generation safe** by switching to an RDF library (rdflib).  
3. **Eliminate the ambiguous comma‑separated keyword literal** and keep a clean, multi‑valued `ex:hasKeyword`.  
4. **Guarantee section uniqueness** by namespacing the section IRIs with the document ID.  
5. **Add a small, self‑contained test harness** that parses the Turtle and asserts the expected predicates.  
6. **Show a quick SPARQL query** that you can run in the UI to confirm everything is now searchable.

---

## 1️⃣ Quick prefix‑fix DONE ALREADY

```diff
@@
-        "@prefix rdfs:<http://www.w3.org/2000/01/rdf-schema#> .",
+        "@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .",
```

Commit that change immediately – it will make the file syntactically correct for any parser.

---

## 2️⃣ Re‑write `build_turtle` with **rdflib**

```python
# doc‑ingestor/main.py
from __future__ import annotations
from typing import Dict, List
import json
from rdflib import Graph, Namespace, Literal, URIRef, RDF
from hashlib import sha1

BASE_IRI = "http://example.org/"

EX = Namespace("http://example.org/")
RDF_NS = RDF

def _safe_literal(text: str, datatype=None):
    """Wrap a string in an rdflib Literal, escaping as needed."""
    return Literal(text, datatype=datatype)

def build_turtle(meta: Dict, sections: List[str], doc_id: str, body: str = "") -> str:
    g = Graph()
    # --- prefixes (they get emitted automatically) ---
    g.bind("ex", EX)
    g.bind("rdf", RDF_NS)
    g.bind("rdfs", Namespace("http://www.w3.org/2000/01/rdf-schema#"))
    g.bind("owl", Namespace("http://www.w3.org/2002/07/owl#"))
    g.bind("xsd", Namespace("http://www.w3.org/2001/XMLSchema#"))

    doc_uri = URIRef(f"{BASE_IRI}Document{doc_id}")
    g.add((doc_uri, RDF_NS.type, EX.Document))

    # --- metadata triples ---
    g.add((doc_uri, EX.hasTitle, _safe_literal(meta["title"])))
    g.add((doc_uri, EX.hasCategory, _safe_literal(meta.get("category", ""))))
    g.add((doc_uri, EX.hasStatus,  _safe_literal(meta.get("status", "Draft"))))
    # body – keep raw string, datatype xsd:string
    g.add((doc_uri, EX.hasContent, _safe_literal(body, datatype=Namespace("http://www.w3.org/2001/XMLSchema#").string)))

    # --- keywords (multi‑valued) ---
    for kw in meta.get("keywords", []):
        g.add((doc_uri, EX.hasKeyword, _safe_literal(kw)))

    # --- sections ---
    for sec in sections:
        sec_hash = sha1(sec.encode("utf-8")).hexdigest()
        sec_uri = URIRef(f"{BASE_IRI}Section_{doc_id}_{sec_hash}")   # doc_id prefix guarantees uniqueness
        g.add((doc_uri, EX.hasSection, sec_uri))

    # --- related artifacts ---
    for rel in meta.get("related", []):
        rel_uri = URIRef(f"{BASE_IRI}Document{rel}")
        g.add((doc_uri, EX.hasRelatedArtifact, rel_uri))

    # Return the serialized Turtle string
    return g.serialize(format="turtle").decode("utf-8")
```

### What changed?

| Old | New | Why |
|-----|-----|-----|
| String concatenation | `rdflib.Graph` | Guarantees correct escaping, datatype handling, and no stray `;`/`.` issues. |
| Single `hasKeywords` literal | Multiple `hasKeyword` triples | Cardinality‑friendly, easier SPARQL. |
| Section URI: `Section_{sha1}` | `Section_{doc_id}_{sha1}` | Prevents cross‑document collisions. |
| Body: `json.dumps(body)[1:-1]` | Raw string with `xsd:string` datatype | Keeps the original body intact; no extra JSON framing. |

---

## 3️⃣ Unit‑test harness (pytest or unittest)

```python
# test_build_turtle.py
import unittest
from pathlib import Path
from rdflib import Graph, URIRef, Literal
from doc_ingestor.main import build_turtle, BASE_IRI, EX

class TestBuildTurtle(unittest.TestCase):
    def setUp(self):
        self.meta = {
            "title": 'Sample "Doc"',
            "category": "Test",
            "status": "Draft",
            "keywords": ["alpha", "beta"],
            "related": ["123"]
        }
        self.sections = ["Intro", "Details"]
        self.body = 'Body with "quotes" and newline\nsecond line.'
        self.turtle = build_turtle(self.meta, self.sections, "001", self.body)

    def test_parse(self):
        g = Graph()
        g.parse(data=self.turtle, format="turtle")

        doc_uri = URIRef(f"{BASE_IRI}Document001")

        # title
        self.assertTrue((doc_uri, EX.hasTitle, Literal('Sample "Doc"')).in_context(g))
        # category
        self.assertTrue((doc_uri, EX.hasCategory, Literal('Test')).in_context(g))
        # status
        self.assertTrue((doc_uri, EX.hasStatus, Literal('Draft')).in_context(g))
        # body
        self.assertTrue((doc_uri, EX.hasContent, Literal(self.body)).in_context(g))
        # keywords
        kw_set = {o.toPython() for _,_,o in g.triples((doc_uri, EX.hasKeyword, None))}
        self.assertSetEqual(kw_set, {"alpha", "beta"})
        # related artifact
        rel_uri = URIRef(f"{BASE_IRI}Document123")
        self.assertTrue((doc_uri, EX.hasRelatedArtifact, rel_uri).in_context(g))
        # sections
        sec_ids = {o.toPython().split("#")[-1] for _,_,o in g.triples((doc_uri, EX.hasSection, None))}
        self.assertEqual(len(sec_ids), 2)

    def test_sparql(self):
        g = Graph()
        g.parse(data=self.turtle, format="turtle")
        q = """
        PREFIX ex: <http://example.org/>
        SELECT ?p WHERE { <http://example.org/Document001> ?p ?o }
        """
        results = list(g.query(q))
        props = {str(row[0]) for row in results}
        expected = {
            "http://example.org/hasTitle",
            "http://example.org/hasCategory",
            "http://example.org/hasStatus",
            "http://example.org/hasContent",
            "http://example.org/hasKeyword",
            "http://example.org/hasSection",
            "http://example.org/hasRelatedArtifact",
        }
        self.assertTrue(expected.issubset(props))

if __name__ == "__main__":
    unittest.main()
```

Run with:

```bash
python -m unittest test_build_turtle.py
```

All tests should pass, confirming that the Turtle string contains every attribute you expect.

---

## 4️⃣ Quick SPARQL check in the UI

```sparql
PREFIX ex: <http://example.org/>
SELECT ?p ?o
WHERE {
  ex:Document001 ?p ?o .
}
```

You should see rows for:

---
ex:hasTitle          "Sample \"Doc\""
ex:hasCategory       "Test"
ex:hasStatus         "Draft"
ex:hasContent        "Body with \"quotes\" and newline\nsecond line."
ex:hasKeyword        "alpha"
ex:hasKeyword        "beta"
ex:hasSection        <http://example.org/Section_001_8238deed4280e27b481b25362606dde509438ffb>
ex:hasSection        <http://example.org/Section_001_...>  # second hash
ex:hasRelatedArtifact <http://example.org/Document123>
---

If any of those lines are missing, the unit test will catch it before you push to production.

---

## 5️⃣ Summary of action items

| ✅ Item | Status |
|---------|--------|
| Add space after `rdfs:` | ✅ (code snippet above) |
| Switch to `rdflib` builder | ✅ (full code block) |
| Remove comma‑separated `hasKeywords` | ✅ (implicit by using multi‑triples) |
| Prefix section IRIs with `doc_id` | ✅ (included in builder) |
| Add unit tests | ✅ (file shown) |


'''

result = run_agent(request)
print("\nFINAL RESPONSE\n")
print(result)


In [ ]:
p = False
for e in events:
    if p:
        print(e.result)
        print("---")
        p = False
    if  "search" in e.result and  "read_file" in e.result:
        p = True
        print(e.result)
        print("---")

In [ ]:
print(['final_response'])

In [ ]:
request = """
yesterday you and i were working on a graph db project that will move us towards having a more persistent learning type experience, we have 
a repo that is taking shape well, we have a collection of documents that we are adding to graph db, and we have a schema that describes documents, proje
cts, people, hypothesises. we set up a llm step in our ingestion of the documents that attempts to extract some meta labels from the project when the ma
rkdown front matter is not quite consistent, overall its looking like the right direction. what i'm wondering right now is that when we add items to gr 
aphdb, we are adding them in a turtle format as described in the doc-ingestor/main.py but i dont see those attributes when i search the documents in the sparkql interface,

## Phase 1 – Discovery (“What’s going wrong?”)

| # | What to check | Why it matters | What the agent should do |
|---|---------------|----------------|--------------------------|
| 1 | **Prefix declarations** | Must be unique & correctly terminated (`.`). | Ask the agent to parse the `prefixes` list and confirm each ends with 
`.` and no duplicate prefixes. |
| 2 | **Triple formatting** | Every triple line must end with a `;` *except* the last one, which must end with `.`. | Have the agent count semicolons 
and verify the last line ends with `.`. |
| 3 | **Escaping of quotes** | Literals containing `"` must be escaped (`\"`). | Request a diff of the generated Turtle vs the raw `meta` to spot 
unescaped quotes. |
| 4 | **Datatype handling** | The `hasKeywords` and `hasRelated` fields are encoded as a single comma‑separated literal – this defeats property 
cardinality. | Ask the agent to suggest a multivalued pattern (e.g., a separate triple for each keyword). |
| 5 | **Missing predicates** | `ex:hasKeyword` is added in a loop, but the first declaration of `ex:hasKeywords` shadows it. | Verify that both 
`hasKeywords` and `hasKeyword` appear with the expected number of triples. |
| 6 | **Section ID collision** | Using `sha1(sec)` may produce duplicate IDs across different docs. | Have the agent check hash collisions in a sample 
set. |
| 7 | **Related artifacts** | The loop that adds `ex:hasRelatedArtifact` adds the triple *after* the last `;` of the previous section, so the line may 
have an extra `;`. | Inspect the string that ends the Turtle to confirm the semicolon is removed correctly. |
| 8 | **Body escaping** | `json.dumps(body)[1:-1]` is an odd trick; it produces a quoted JSON string *without* outer quotes, but any internal quotes 
will be escaped. | Test the agent on a body that contains special chars to see if the literal is parsed correctly. |
| 9 | **URI validity** | The constructed IRIs (`<BASE_IRI>Document{doc_id}`) must not contain illegal characters. | Have the agent validate that each 
URI conforms to RFC 3987. |
|10 | **SPARQL endpoint mapping** | If the graph is stored in a triple store that auto‑generates indexes, it may ignore properties that are not 
explicitly typed. | Ask the agent to run a SPARQL query like `SELECT ?p WHERE { <BASE_IRI>Document{doc_id}> ?p ?o }` and list the properties actually 
present. |

**What the agent should produce in this phase**

* A **diagnostic report** (plain text or Markdown) summarizing each of the 10 checks, highlighting any failures, and listing the raw Turtle snippet 
that is problematic.  
* If the agent identifies a missing predicate or malformed triple, it should mark it for the next phase.


"""

result = run_agent(request)
print("\nFINAL RESPONSE\n")
print(result)


In [ ]:
request = """
**🚀 Legible Delegation Prompt – “Markdown ↔ GraphDB” Ingestion Service**

---

### 1️⃣ Purpose  
Build a lightweight, manual‑trigger Docker container that

1. Clones the **CAE docs repo** (GitHub SSH URL you supplied).  
2. Parses every markdown file to extract:
   * Header fields: `title`, `category`, `status`, `keywords`, `related`
   * All section titles (e.g., `## Intent`, `## Purpose`, …)  
3. Serialises each file as a small **Turtle fragment** that represents an `ex:Document` instance and its metadata.  
4. POSTs the fragment to the **GraphDB inference‑backbone** repository (`http://graphdb:7200/repositories/inference-backbone` by default).  
5. Exposes a simple health‑check and logs progress to stdout.

> **Use‑case**: Run `docker compose up doc-ingestor` whenever you want a fresh snapshot of the docs in the graph. No auto‑deployment – perfect for the 
first pass.

---

### 2️⃣ Inputs (to be supplied by the next agent)

| Variable | Default / Example | What to override? |
|----------|-------------------|-------------------|
| `REPO_URL` | `git@github.com:forjonathanwilsonyahoocom/cae.git` | Your repo URL (SSH). |
| `CLONE_DIR` | `/data/docs` | Where the repo is cloned inside the container. |
| `GRAPHDB_URL` | `http://graphdb:7200/repositories/inference-backbone` | Endpoint to POST Turtle statements. |
| `ONTOLOGY_PREFIX` | `http://example.org/` | Base namespace for the `ex:` prefix. |
| `GRAPHDB_REPO` | `inference-backbone` | Repository ID (used in the URL). |

---

### 3️⃣ Core Files to Create

| File | Purpose | Key Dependencies |
|------|---------|-------------------|
| **Dockerfile** | Build image with Python 3.12 and `git`. | `python:3.12-slim`, `git`, `pip install` |
| **requirements.txt** | `gitpython`, `pyyaml`, `requests` | |
| **entrypoint.sh** | Entrypoint script: clone repo → parse → ingest | Uses `git clone`, `python main.py` |
| **main.py** | Parses markdown → builds Turtle → POSTs | Uses `gitpython`, `pyyaml`, `requests` |
| **README.md** | How to run the container | |
| **.dockerignore** | Ignore large Git history | |

---

### 4️⃣ Example Turtle Skeleton (to be generated by `main.py`)

```turtle
@prefix ex:   <http://example.org/> .
@prefix xsd:  <http://www.w3.org/2001/XMLSchema#> .

ex:Document123 a ex:Document ;
    ex:hasTitle "My Artifact" ;
    ex:hasCategory "Concept" ;
    ex:hasStatus "Draft" ;
    ex:hasKeyword "AI" ;
    ex:hasKeyword "Inference" ;
    ex:hasSection ex:Section_Intent, ex:Section_Purpose ;
    ex:hasRelatedArtifact ex:Document456 .

ex:Section_Intent a ex:Section ;
    ex:sectionTitle "Intent" ;
    ex:sectionOrder 1 .

# ... etc. ...
```

*The script will auto‑generate IRIs (`ex:Document<hash>`), section IRIs, and optional `hasRelatedArtifact` edges.*

---

### 5️⃣ Pseudo‑code for `main.py`

```python
import os, pathlib, hashlib, yaml, json, requests, git, textwrap
from pathlib import Path

BASE_IRI = os.getenv('ONTOLOGY_PREFIX', 'http://example.org/')

def sha1(text):
    return hashlib.sha1(text.encode()).hexdigest()

def parse_markdown(md_path: Path):
    content = md_path.read_text()
    # split header and body
    header, body = content.split('---', 1)
    meta = yaml.safe_load(header)
    # Sections: every '## ' line
    sections = [s.strip() for s in body.split('\n') if s.startswith('## ')]
    return meta, sections

def build_turtle(meta, sections, doc_id):
    parts = [
        f"<{BASE_IRI}Document{doc_id}> a ex:Document ;",
        f"  ex:hasTitle \"{meta['title']}\" ;",
        f"  ex:hasCategory \"{meta.get('category', '')}\" ;",
        f"  ex:hasStatus \"{meta.get('status', '')}\" ;",
    ]
    for kw in meta.get('keywords', []):
        parts.append(f"  ex:hasKeyword \"{kw}\" ;")
    # sections
    for idx, sec in enumerate(sections, 1):
        sec_iri = f"Section_{sha1(sec)}"
        parts.append(f"  ex:hasSection <{BASE_IRI}{sec_iri}> ;")
    # related
    for rel in meta.get('related', []):
        parts.append(f"  ex:hasRelatedArtifact <{BASE_IRI}Document{rel}> ;")
    # close last triple
    parts[-1] = parts[-1].rstrip(' ;') + " ."
    return "\n".join(parts)

def post_turtle(turtle_str):
    url = os.getenv('GRAPHDB_URL')
    headers = {'Content-Type': 'text/turtle'}
    r = requests.post(url, data=turtle_str.encode(), headers=headers)
    r.raise_for_status()

def main():
    repo_dir = Path(os.getenv('CLONE_DIR', '/data/docs'))
    # ensure repo exists
    if not repo_dir.exists():
        git.Repo.clone_from(os.getenv('REPO_URL'), repo_dir)
    # walk .md files
    for md_path in repo_dir.rglob('*.md'):
        meta, sections = parse_markdown(md_path)
        doc_id = sha1(md_path.as_posix())
        turtle = build_turtle(meta, sections, doc_id)
        post_turtle(turtle)
        print(f"Ingested {md_path}")

if __name__ == "__main__":
    main()
```

> *Adjust the header parsing if your YAML front‑matter uses a different delimiter.*

---

### 6️⃣ Docker Compose Snippet (to be added by the next agent)

```yaml
services:
  doc-ingestor:
    build: ./doc-ingestor
    environment:
      REPO_URL: "git@github.com:forjonathanwilsonyahoocom/cae.git"
      CLONE_DIR: "/data/docs"
      GRAPHDB_URL: "http://graphdb:7200/repositories/inference-backbone"
      ONTOLOGY_PREFIX: "http://example.org/"
    volumes:
      - ./cae:/data/docs:ro   # optional, if you want to keep the repo locally
```

---

### 7️⃣ Clarifying Questions for the Next Agent

1. **Ontology** – Do you already have an `ex:Document`/`ex:Section` class defined in your `schema.ttl`? If not, the script will create them on the 
fly.  
2. **Section granularity** – Should we store every `## ` title as a separate node, or just keep them as literals?  
3. **Related artifacts** – The `related:` list in the header is expected to contain *document IDs* (the hash used for the `ex:Document` IRI). If you 
prefer relative paths or file names, let me know.  
4. **Authentication** – The container will `git clone` over SSH. Do you have a key mounted (`/root/.ssh/id_rsa`) or should we use HTTPS?  
5. **GraphDB repository** – Confirm that the default `inference-backbone` is the correct repo; otherwise adjust `GRAPHDB_URL`.  



"""

result = run_agent(request)
print("\nFINAL RESPONSE\n")
print(result)


In [ ]:
print(list_files.invoke({}))


In [ ]:
import pprint
print(len(events))
pprint.pprint(events)